# Hemodynamic Correction LSTM

This notebook implements the LSTM-based hemodynamic artifact correction described in:
> **Inferring norepinephrine dynamics from partial observations reveals the temporal structure of elevations during arousal — Erin Neyhart, Brandon R. Munn, Na Zhou, Peilin Yang, Jiesi Feng, Yulong Li, James M. Shine, Jacob Reimer**

## Overview

Two-photon fluorescence recordings of rNE0.5 (a GRAB, or G-protein based, norepinephrine sensor) can be
contaminated by hemodynamic artifacts that are represented by the activity of an inert fluorophore (the 'mut' control). When this simultaneous control channel is unavailable, we train an LSTM to predict the hemodynamic contribution to the GRAB signal from the NE trace itself plus optional behavioral and brightness-binned predictors, enabling post-hoc correction.

## Four predictor sets (one model each)

| Model | Inputs | Input size |
|-------|--------|------------|
| 0 | rNE fluorescence only | 1 |
| 1 | rNE + brightness bins (10 spatial bins) | 11 |
| 2 | rNE + behavioral variables (pupil, velocity + derivatives) | 5 |
| 3 | rNE + brightness bins + behavioral variables | 15 |

## Notebook structure

1. Imports and configuration
2. Data loading and preprocessing
3. Model architecture
4. Training (stratified train/test split by recording)
5. Evaluation and visualization
6. Save / load model weights

## Data

Input CSVs are expected in `DATA_DIR` with the naming convention `{brain_area}_{animal_id}_{session}_{scan_idx}.csv`.
Each CSV contains the following columns:

| Column | Description |
|--------|-------------|
| `fluorescence_timestamps` | Fluorescence frame times (s) |
| `rNE_fluorescence` | Raw rNE0.5 fluorescence trace |
| `mut_fluorescence` | Raw inert control (mut) fluorescence trace |
| `velocity_in_cm_per_sec` | Treadmill speed |
| `velocity_timestamps` | Treadmill timestamps (s) |
| `radius_in_pixels` | Pupil radius |
| `radius_timestamps` | Pupil timestamps (s) |

A second CSV per recording (`{brain_area}_{animal_id}_{session}_{scan_idx}_BrightnessBins.csv`) contains:

| Column | Description |
|--------|-------------|
| `timestamps` | Timestamps (s) |
| `Mask_1` … `Mask_10` | Mean fluorescence in 10 spatial brightness bins |

See `data/README.md` for the dataset DOI and download instructions.


## 1. Imports and configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import random
from collections import defaultdict
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from utils import (
    find_nearest,
    interpolate_to_new_timestamps,
    interpolate_nans,
    preprocess_dFF,
    preprocess_no_dFF,
    lowpass_butter,
    median_filter,
    zscore,
    safe_zscore,
)

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
# ---------------------------------------------------------------------------
# Configuration — edit these paths before running
# ---------------------------------------------------------------------------

# Directory containing the per-recording CSVs
DATA_DIR = "data/hemodynamic_prediction/rNE-mut_prediction_data/"

# Directory containing the brightness-bin CSVs
BIN_DIR = "data/hemodynamic_prediction/rNE-BrightnessBins/"

# Directory where model weights and train/test data will be saved
OUTPUT_DIR = "output/"

# Preprocessing parameters
LOWPASS_CUTOFF = 1.0      # Hz — lowpass filter cutoff for fluorescence
TARGET_FPS     = 20.0     # Hz — common sampling rate (pupil camera rate)
WINDOW_SEC     = 2 * 60   # s  — length of each training chunk (2 minutes)
MAX_RATIO      = 5        # dF/F guard: skip traces with range/5th-%ile > this

# Training parameters
N_EPOCHS   = 50
BATCH_SIZE = 32
LR         = 1e-3

HIDDEN_SIZE = 256
FC_SIZE     = 256
DROPOUT     = 0.1

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Data loading and preprocessing

Each recording is loaded from CSV, preprocessed to dF/F, and split into
non-overlapping 2-minute chunks. The target for the LSTM is the mut
(inert control) fluorescence, which represents the hemodynamic artifact.

Recordings with a negative raw baseline are skipped (as dF/F cannot be calculated with a negative baseline).

Each element of `alldat` is a list:
```
[mut_chunk, rNE_chunk, brightness_bins_chunk, behavior_chunk, recording_name]
```
where `behavior_chunk` has shape `(4, n_timepoints)` with rows:
`[pupil_radius, d/dt pupil_radius, velocity, d/dt velocity]`.

In [3]:
# Build the list of recording filenames from the data directory.
# Each entry is the base filename (without extension), e.g. 'M1_12345_3_1'.
import glob
main_csvs = glob.glob(os.path.join(DATA_DIR, "*.csv"))
recording_names = [
    os.path.splitext(os.path.basename(f))[0]
    for f in main_csvs
    if not f.endswith("_BrightnessBins.csv")
]
print(f"Found {len(recording_names)} recordings.")

Found 30 recordings.


In [4]:
alldat = []

for filename in recording_names:

    # --- Load main recording CSV ---
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, filename + ".csv"))
    except:
        print('no main file ' + str(filename))
        continue

    fl_times     = df['fluorescence_timestamps'].values
    sensor       = df['rNE_fluorescence'].values
    control      = df['mut_fluorescence'].values
    velocity     = df['velocity_in_cm_per_sec'].values
    vel_times    = df['velocity_timestamps'].values
    radius       = df['radius_in_pixels'].values
    radius_times = df['radius_timestamps'].values

    # --- Load brightness-bin CSV ---
    try:
        df_bins = pd.read_csv(os.path.join(BIN_DIR, filename + "_BrightnessBins.csv"))
    except:
        print('no binned file ' + str(filename))
        continue

    binned_times = df_bins['timestamps'].values
    binned_F     = df_bins[[f"Mask_{i}" for i in range(1, 11)]].values  # Mask_1 … Mask_10

    # cut/align
    scan_start_secs = fl_times[np.isfinite(fl_times)][0] + 20
    scan_stop_secs  = fl_times[np.isfinite(fl_times)][-1]

    start = find_nearest(fl_times, scan_start_secs)
    stop  = find_nearest(fl_times, scan_stop_secs)
    fl_times = fl_times[start:stop]
    sensor   = sensor[start:stop]
    control  = control[start:stop]

    start     = find_nearest(vel_times, scan_start_secs)
    stop      = find_nearest(vel_times, scan_stop_secs)
    vel_times = vel_times[start:stop]
    velocity  = velocity[start:stop]

    start        = find_nearest(radius_times, scan_start_secs)
    stop         = find_nearest(radius_times, scan_stop_secs)
    radius_times = radius_times[start:stop]
    radius       = radius[start:stop]

    if np.any(np.isnan(radius_times)):
        radius_times = interpolate_nans(radius_times.copy())

    start        = find_nearest(binned_times, scan_start_secs)
    stop         = find_nearest(binned_times, scan_stop_secs)
    binned_times = binned_times[start:stop]
    binned_F     = binned_F[start:stop]

    # preprocess
    fps      = 1.0 / np.nanmedian(np.diff(fl_times))
    pupilfps = round(1.0 / np.nanmedian(np.diff(radius_times)))

    # detrend, filter, normalize, interpolate
    try:
        sensor  = preprocess_dFF(sensor,  fps, 5, filt=True, interp=True, lowpass_cutoff=LOWPASS_CUTOFF, old_times=fl_times, new_times=radius_times)
        control = preprocess_dFF(control, fps, 5, filt=True, interp=True, lowpass_cutoff=LOWPASS_CUTOFF, old_times=fl_times, new_times=radius_times)
    except:
        print('negative baseline ' + str(filename))
        continue

    radius    = lowpass_butter(radius, pupilfps, LOWPASS_CUTOFF, order=5, linear_interp=True)
    # note: pixel-to-mm conversion omitted — radius is z-scored before use,
    # so absolute units do not affect model inputs
    radius_dv = np.gradient(radius)

    # if np.any(np.isnan(vel_times)): vel_times = interpolate_nans(vel_times.copy())
    if np.any(np.isnan(velocity)): velocity = interpolate_nans(velocity.copy())
    velocity    = median_filter(abs(velocity), 1/np.nanmedian(np.diff(vel_times)), 0.5)
    velocity    = interpolate_to_new_timestamps(vel_times, velocity, radius_times)
    velocity_dv = np.gradient(velocity)

    # detrend, filter, and interpolate each bin independently
    binned_F = np.stack([
        preprocess_no_dFF(binned_F[:, i], fps, filt=True, interp=True,
                          lowpass_cutoff=LOWPASS_CUTOFF,
                          old_times=binned_times,
                          new_times=radius_times)
        for i in range(binned_F.shape[1])
    ], axis=1)  # shape: (time, 10)

    # global joint zscore across all bins together
    all_bins_mean = np.nanmean(binned_F)
    all_bins_std  = np.nanstd(binned_F)
    binned_F = (binned_F - all_bins_mean) / (all_bins_std + 1e-8)

    # cut into 2-min chunks
    ndt    = int(WINDOW_SEC * pupilfps)
    n2min  = int(len(radius_times) // ndt)

    for NN in range(n2min):
        start = NN * ndt
        end   = (NN + 1) * ndt
        tclip = slice(start, end)

        mut_chunk    = control[tclip]
        red_chunk    = sensor[tclip]
        binned_chunk = binned_F[tclip, :]
        beh_chunk    = np.vstack([
            zscore(radius[tclip]),
            zscore(radius_dv[tclip]),
            zscore(velocity[tclip]),
            zscore(velocity_dv[tclip])
        ])  # shape: (4, ndt)

        recname = filename
        alldat.append([mut_chunk, red_chunk, binned_chunk, beh_chunk, recname])

print(f"Total 2-min chunks: {len(alldat)}")

negative baseline V1_27195_3_1
Total 2-min chunks: 339


## 3. Model architecture

Single-layer LSTM followed by two fully connected layers with ReLU and dropout.
The model predicts the hemodynamic (mut) signal at every time step.

In [12]:
class LSTMRegressor(nn.Module):
    """LSTM-based sequence-to-sequence regressor.

    Parameters
    ----------
    input_size : int
        Number of input features per time step.
    hidden_size : int
        LSTM hidden state size.
    fc_size : int
        Size of the intermediate fully connected layer.
    dropout_p : float
        Dropout probability applied between FC layers.
    output_size : int
        Output features per time step (1 for scalar regression).
    """

    def __init__(self, input_size, hidden_size=256, fc_size=256,
                 dropout_p=0.1, output_size=1):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc1     = nn.Linear(hidden_size, fc_size)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2     = nn.Linear(fc_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)           # (batch, seq_len, hidden)
        out = torch.relu(self.fc1(out)) # (batch, seq_len, fc_size)
        out = self.dropout(out)
        out = self.fc2(out)             # (batch, seq_len, 1)
        return out.squeeze(-1)          # (batch, seq_len)

## 4. Training

The dataset is split into train and test sets by **recording** (not by chunk),
stratified by brain area. This prevents data leakage from the same animal
appearing in both train and test sets.

Four models are trained in parallel, one per predictor set.

In [13]:
# ---------------------------------------------------------------------------
# Train/test split: stratified by brain area, held out at the recording level
# ---------------------------------------------------------------------------
total_num    = len(alldat)
all_recnames = [alldat[i][4] for i in range(total_num)]
unique_recs  = list(set(all_recnames))

# Group recordings by brain area (first element of filename, e.g. 'M1')
recs_by_area = defaultdict(list)
for recname in unique_recs:
    area = recname.split("_")[0]
    recs_by_area[area].append(recname)

# Hold out ~20% of recordings per area
leaveOutRecs = set()
for area, recs in recs_by_area.items():
    n_test = max(1, round(0.2 * len(recs)))
    leaveOutRecs.update(random.sample(recs, n_test))

print("Held-out recordings per brain area:")
for area, recs in recs_by_area.items():
    held = [r for r in recs if r in leaveOutRecs]
    print(f"  {area}: {held}")
print(f"\nTotal held out: {len(leaveOutRecs)} / {len(unique_recs)} recordings")

Held-out recordings per brain area:
  V1: ['V1_27135_6_1']
  M1: ['M1_27411_3_4', 'M1_27032_5_3', 'M1_27411_1_1']
  SS1: ['SS1_27195_12_3', 'SS1_27195_5_1']

Total held out: 6 / 29 recordings


In [ ]:
# ---------------------------------------------------------------------------
# Build predictor arrays and train all four models
# ---------------------------------------------------------------------------
criterion = nn.MSELoss()

# Input sizes: [rNE only, rNE+bins, rNE+beh, rNE+bins+beh]
INPUT_SIZES = [1, 11, 5, 15]
PREDICTOR_LABELS = ["rNE", "rNE + brightness bins",
                    "rNE + behavior", "rNE + bins + behavior"]

models, optimizers = [], []
X_trains, y_trains = [], []
X_tests,  y_tests  = [], []

for dd in range(4):
    train_x, train_y, test_x, test_y = [], [], [], []

    for i in range(total_num):
        recname = alldat[i][4]
        mut     = alldat[i][0]          # target
        rNE     = alldat[i][1]          # NE sensor
        bins    = alldat[i][2]          # brightness bins, shape (ndt, 10)
        beh     = alldat[i][3].T        # behavioral, shape (ndt, 4)

        if dd == 0:
            predictor = rNE[:, None]                      # (ndt, 1)
        elif dd == 1:
            predictor = np.hstack([rNE[:, None], bins])   # (ndt, 11)
        elif dd == 2:
            predictor = np.hstack([rNE[:, None], beh])    # (ndt, 5)
        else:
            predictor = np.hstack([rNE[:, None], bins, beh])  # (ndt, 15)

        if recname in leaveOutRecs:
            test_x.append(predictor)
            test_y.append(mut)
        else:
            train_x.append(predictor)
            train_y.append(mut)

    X_train = np.stack(train_x, axis=0)   # (N_train, ndt, n_features)
    y_train = np.stack(train_y, axis=0)   # (N_train, ndt)
    X_test  = np.stack(test_x,  axis=0)
    y_test  = np.stack(test_y,  axis=0)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

    X_trains.append(X_train_t);  y_trains.append(y_train_t)
    X_tests.append(X_test_t);    y_tests.append(y_test_t)

    loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=BATCH_SIZE, shuffle=True
    )

    model = LSTMRegressor(
        input_size=X_train.shape[-1],
        hidden_size=HIDDEN_SIZE, fc_size=FC_SIZE, dropout_p=DROPOUT
    )
    optimizer = optim.Adam(model.parameters(), lr=LR)
    models.append(model)
    optimizers.append(optimizer)

    print(f"\n=== Model {dd}: {PREDICTOR_LABELS[dd]} ===")
    for epoch in range(N_EPOCHS):
        model.train()
        total_loss = 0
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{N_EPOCHS}  "
                  f"loss={total_loss/len(loader):.4f}")


=== Model 0: rNE ===
  Epoch  10/50  loss=0.0012


In [ ]:
# Save model weights and train/test split
for i, model in enumerate(models):
    torch.save(
        model.state_dict(),
        os.path.join(OUTPUT_DIR, f"mut_model_{i}_weights.pth")
    )

save_data = {
    "X_trains":     [X.cpu().numpy() for X in X_trains],
    "y_trains":     [y.cpu().numpy() for y in y_trains],
    "X_tests":      [X.cpu().numpy() for X in X_tests],
    "y_tests":      [y.cpu().numpy() for y in y_tests],
    "leaveOutRecs": list(leaveOutRecs),
}
with open(os.path.join(OUTPUT_DIR, "mut_model_train_test_data.pkl"), "wb") as f:
    pickle.dump(save_data, f)

print("Model weights and train/test data saved.")

## 5. Evaluation

We evaluate each model using Pearson correlation and RMSE on held-out
chunks. Results are shown as box-and-swarm plots, mirroring the paper figures.

In [ ]:
def compute_metrics(model, X, y):
    """Per-chunk Pearson r and RMSE for a model on dataset (X, y)."""
    model.eval()
    corrs, rmses = [], []
    with torch.no_grad():
        for i in range(X.shape[0]):
            y_true = y[i].numpy()
            y_pred = model(X[i].unsqueeze(0)).squeeze(0).numpy()
            if np.all(np.isnan(y_true)) or np.all(np.isnan(y_pred)):
                continue
            r, _   = pearsonr(y_true, y_pred)
            rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
            corrs.append(r)
            rmses.append(rmse)
    return np.array(corrs), np.array(rmses)


train_corrs, train_rmses = [], []
test_corrs,  test_rmses  = [], []

for dd in range(4):
    tr_c, tr_r = compute_metrics(models[dd], X_trains[dd], y_trains[dd])
    te_c, te_r = compute_metrics(models[dd], X_tests[dd],  y_tests[dd])
    train_corrs.append(tr_c);  train_rmses.append(tr_r)
    test_corrs.append(te_c);   test_rmses.append(te_r)
    print(f"Model {dd} ({PREDICTOR_LABELS[dd]})")
    print(f"  Train  r={np.median(tr_c):.3f}  RMSE={np.median(tr_r):.4f}")
    print(f"  Test   r={np.median(te_c):.3f}  RMSE={np.median(te_r):.4f}")

In [ ]:
def box_swarm(ax, data_list, title, ylabel=None):
    """Box plot with overlaid individual data points."""
    ax.boxplot(data_list, positions=np.arange(1, 5), widths=0.6)
    for i, data in enumerate(data_list):
        x_jitter = np.random.normal(i + 1, 0.04, size=len(data))
        ax.plot(x_jitter, data, "o", alpha=0.4, markersize=4)
    ax.set_xticks(np.arange(1, 5))
    ax.set_xticklabels(["rNE", "+bins", "+beh", "+bins\n+beh"], rotation=0)
    ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", alpha=0.4)


fig, axs = plt.subplots(2, 2, figsize=(10, 8))
box_swarm(axs[0, 0], train_corrs, "Training — Pearson r",  ylabel="r")
box_swarm(axs[0, 1], train_rmses, "Training — RMSE")
box_swarm(axs[1, 0], test_corrs,  "Test — Pearson r",      ylabel="r")
box_swarm(axs[1, 1], test_rmses,  "Test — RMSE")
plt.suptitle("Hemodynamic correction LSTM: performance by predictor set")
plt.tight_layout()
plt.show()

In [ ]:
# Example trace plots: predicted vs actual mut signal for the best model (model 3)
model = models[3]
model.eval()

n_examples = min(5, X_tests[3].shape[0])
fig, axes = plt.subplots(n_examples, 1, figsize=(12, 3 * n_examples), sharex=True)

with torch.no_grad():
    for i, ax in enumerate(axes):
        y_true = y_tests[3][i].numpy()
        y_pred = model(X_tests[3][i].unsqueeze(0)).squeeze(0).numpy()
        t = np.arange(len(y_true)) / TARGET_FPS
        ax.plot(t, y_true, color="black",  label="Actual (mut)",   linewidth=1.2)
        ax.plot(t, y_pred, color="tomato", label="Predicted (LSTM)", linewidth=1.2,
                linestyle="--")
        ax.set_ylabel("dF/F")
        if i == 0:
            ax.legend()

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Model 3 (rNE + bins + behavior): test set predictions")
plt.tight_layout()
plt.show()

## 6. Load saved weights (for inference without retraining)

The cell below shows how to reload the trained models. This is what you
would run to apply the correction to new recordings without repeating training.

In [ ]:
# Reload models from saved weights
INPUT_SIZES = [1, 11, 5, 15]
loaded_models = []

for i, input_size in enumerate(INPUT_SIZES):
    m = LSTMRegressor(
        input_size=input_size,
        hidden_size=HIDDEN_SIZE, fc_size=FC_SIZE, dropout_p=DROPOUT
    )
    m.load_state_dict(
        torch.load(os.path.join(OUTPUT_DIR, f"mut_model_{i}_weights.pth"),
                   map_location="cpu")
    )
    m.eval()
    loaded_models.append(m)

print(f"Loaded {len(loaded_models)} models.")